# The sidebar: folders from strings or from data

Every layer files into a tree of folders in the map's sidebar. Paths come from
literal strings, from columns in your data, or both mixed — and folders can act as
checkboxes or radio groups.

In [ ]:
import numpy as np
import pandas as pd
from swiftmap import Map

rng = np.random.default_rng(5)
n = 150
df = pd.DataFrame({
    "lat": 36.02 + rng.normal(0, 0.06, n),
    "lon": -5.45 + rng.normal(0, 0.10, n),
    "station": rng.choice(["Alpha", "Bravo", "Charlie", "Delta"], n),
    "region": rng.choice(["East", "West"], n),
    "status": rng.choice(["Active", "Idle"], n, p=[0.7, 0.3]),
    "reading": np.round(rng.gamma(4, 4, n), 1),
})

## Literal paths

A `/` in the string nests folders:

In [ ]:
m = Map()
m.add_circle_markers(df[df.status == "Active"], name="Active",
                     layer_group="Feeds/Live")
m.add_circle_markers(df[df.status == "Idle"], name="Idle",
                     layer_group="Feeds/Standby", color="#bab0ac")
m

## Paths from the data

Any part of `layer_group` that matches a column resolves **per row**, and `name`
can come from a column the same way — rows sharing a resolved path and name merge
into one layer. One call, a whole tree:

In [ ]:
m = Map()
m.add_circle_markers(df, name="station",
                     layer_group=["Sensors", "region", "status"],
                     color_col="status")
m

## Configuring folders

`configure_group` takes a folder path and sets its behavior — `collapsed`,
`visible`, and `multi_select=False` for mutually exclusive radio buttons.
`group_multi_select=False` on any `add_*` call is the same thing said inline.

In [ ]:
m.configure_group("Sensors", collapsed=False)
m.configure_group("Sensors/East", multi_select=False)   # radios: one at a time
m.configure_group("Sensors/West/Idle", visible=False);

## Basemaps are a group like any other

The default Basemaps folder is already a radio group. Add presets or any
`{z}/{x}/{y}` tile URL:

In [ ]:
m2 = Map()
m2.add_basemap("Positron")
m2.add_basemap("https://{s}.tile.opentopomap.org/{z}/{x}/{y}.png",
               attribution="&copy; OpenTopoMap", layer_group="Basemaps")
m2.add_circle_markers(df, name="Sensors")
m2

## Collections merge under one entry

Layers born from one `add_collection` call share a name on purpose: they collapse
into a single sidebar entry with the geometries as children. Telling the children
apart programmatically is what `types=` targeting is for — **05_layer_control**.

In [ ]:
wkt_df = pd.DataFrame({
    "geometry": [
        "POINT (-5.36 36.13)",
        "LINESTRING (-5.44 36.05, -5.36 36.09, -5.30 36.14)",
        "POLYGON ((-5.42 36.00, -5.32 36.00, -5.32 36.06, "
        "-5.42 36.06, -5.42 36.00))",
    ],
})
m3 = Map()
m3.add_collection(wkt_df, name="Survey", layer_group="Field Data")
m3